In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input/workdata'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/workdata/fake/id16_id3_0003/frame_0.jpg
/kaggle/input/workdata/fake/id16_id3_0003/frame_1.jpg
/kaggle/input/workdata/fake/id48_id41_0003/frame_0.jpg
/kaggle/input/workdata/fake/id48_id41_0003/frame_1.jpg
/kaggle/input/workdata/fake/id27_id28_0008/frame_0.jpg
/kaggle/input/workdata/fake/id27_id28_0008/frame_1.jpg
/kaggle/input/workdata/fake/id41_id45_0001/frame_0.jpg
/kaggle/input/workdata/fake/id41_id45_0001/frame_1.jpg
/kaggle/input/workdata/fake/id53_id50_0005/frame_0.jpg
/kaggle/input/workdata/fake/id53_id50_0005/frame_1.jpg
/kaggle/input/workdata/fake/id53_id50_0005/frame_2.jpg
/kaggle/input/workdata/fake/id38_id30_0002/frame_0.jpg
/kaggle/input/workdata/fake/id38_id30_0002/frame_1.jpg
/kaggle/input/workdata/fake/id3_id0_0000/frame_0.jpg
/kaggle/input/workdata/fake/id3_id0_0000/frame_1.jpg
/kaggle/input/workdata/fake/id3_id0_0000/frame_2.jpg
/kaggle/input/workdata/fake/id45_id48_0009/frame_0.jpg
/kaggle/input/workdata/fake/id45_id48_0009/frame_1.jpg
/kaggle/input/work

In [ ]:
# 1. GPU SETUP
try:
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"Configured {len(gpus)} GPUs.")
    
    # Enable Mixed Precision
    policy = mixed_precision.Policy('mixed_float16')
    mixed_precision.set_global_policy(policy)
    
except RuntimeError:
    pass # Already initialized

Configured 2 GPUs.


In [6]:
# System & Helper Libraries
import os
import glob
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.utils import shuffle

# Image Processing 
import cv2
from PIL import Image, ImageEnhance

# TensorFlow & Keras Core
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras import Input

#  Keras Layers 
from tensorflow.keras.layers import (
    Conv2D, Dense, Flatten, GlobalAveragePooling1D, 
    Reshape, Add, LayerNormalization, MultiHeadAttention, 
    Lambda, Dropout, BatchNormalization, Activation
)

# Optimizers & Preprocessing
from tensorflow.keras.optimizers import Adam, SGD, RMSprop
from tensorflow.keras.preprocessing.image import load_img, img_to_array

# Pre-trained Models 
from tensorflow.keras.applications import VGG16          
from tensorflow.keras.applications import MobileNetV3Small,ResNet50

# Configuration
physical_devices = tf.config.list_physical_devices('GPU')
if len(physical_devices) > 0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)
    print(f"GPU Detected: {physical_devices[0].name}")
else:
    print("No GPU detected.")
print(f"TensorFlow Version: {tf.__version__}")

GPU Detected: /physical_device:GPU:0
TensorFlow Version: 2.19.0


In [7]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, applications, mixed_precision
from glob import glob
from sklearn.utils import shuffle

In [9]:
REAL_PATH = "/kaggle/input/workdata/real"
FAKE_PATH = "/kaggle/input/workdata/fake"
SEQ_LENGTH = 10
IMG_SIZE = 224
BATCH_SIZE = 8  # Lowered slightly to handle the bigger model

In [10]:
def augment_frame(img):
    """
    Randomly applies flips and slight brightness changes.
    This prevents the model from memorizing the 590 real videos.
    """
    # 50% chance to flip horizontally
    if random.random() > 0.5:
        img = cv2.flip(img, 1)
    
    # Slight brightness adjustment
    value = random.uniform(0.8, 1.2)
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    hsv = np.array(hsv, dtype=np.float64)
    hsv[:, :, 2] = hsv[:, :, 2] * value
    hsv[:, :, 2][hsv[:, :, 2] > 255] = 255
    hsv = np.array(hsv, dtype=np.uint8)
    img = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)
    
    return img

In [ ]:
# 3. RECURSIVE LOADER
def load_data_recursive(root_dir, label):
    data_list = []
    for root, dirs, files in os.walk(root_dir):
        images = sorted([f for f in files if f.lower().endswith(('.jpg', '.png', '.jpeg'))])
        if len(images) > 0:
            video_id = os.path.basename(root)
            full_paths = [os.path.join(root, img) for img in images]
            data_list.append((video_id, label, full_paths))
    return data_list

print("Loading dataset...")
real_data = load_data_recursive(REAL_PATH, 0)
fake_data = load_data_recursive(FAKE_PATH, 1)
full_dataset = shuffle(real_data + fake_data)

print(f"Found {len(real_data)} Real, {len(fake_data)} Fake.")

⏳ Loading dataset...
✅ Found 590 Real, 5639 Fake.


In [ ]:
# 4. ADVANCED GENERATOR
class ProVideoGenerator(tf.keras.utils.Sequence):
    def __init__(self, dataset, batch_size, seq_length, augment=True):
        self.dataset = dataset
        self.batch_size = batch_size
        self.seq_length = seq_length
        self.augment = augment
        self.img_size = (IMG_SIZE, IMG_SIZE)
        
    def __len__(self):
        return len(self.dataset) // self.batch_size

    def __getitem__(self, index):
        batch = self.dataset[index * self.batch_size : (index + 1) * self.batch_size]
        X, y = [], []
        
        for vid_id, label, image_paths in batch:
            if not image_paths: continue
            
            frames = []
            # Loop for short videos
            while len(frames) < self.seq_length:
                for p in image_paths:
                    if len(frames) >= self.seq_length: break
                    try:
                        img = cv2.imread(p)
                        if img is not None:
                            img = cv2.resize(img, self.img_size)
                            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                            
                            # APPLY AUGMENTATION ONLY FOR TRAINING
                            if self.augment:
                                img = augment_frame(img)
                                
                            img = img / 255.0
                            frames.append(img)
                    except: pass
            
            if len(frames) == self.seq_length:
                X.append(frames)
                y.append(label)
                
        if len(X) == 0:
            return np.zeros((self.batch_size, self.seq_length, *self.img_size, 3)), np.zeros(self.batch_size)
            
        return np.array(X), np.array(y)

train_gen = ProVideoGenerator(full_dataset, BATCH_SIZE, SEQ_LENGTH, augment=True)

In [ ]:
# 5. CLASS WEIGHTS
num_real = len(real_data)
num_fake = len(fake_data)
total = num_real + num_fake
weight_0 = (1 / (num_real + 1e-6)) * (total / 2.0)
weight_1 = (1 / (num_fake + 1e-6)) * (total / 2.0)
class_weights = {0: weight_0, 1: weight_1}

In [ ]:
# 6. THE PRO MODEL (Bidirectional + Deep Fine Tuning)
def build_pro_model():
    base_model = applications.ResNet50V2(include_top=False, weights='imagenet', pooling='avg')
    
    # UNFREEZE MORE LAYERS (The "conv5" block)
    base_model.trainable = True
    # Freeze the first 140 layers, keep the last ~50 trainable
    # This allows the model to learn "skin textures" better
    for layer in base_model.layers[:-50]:
        layer.trainable = False
        
    inputs = layers.Input(shape=(SEQ_LENGTH, IMG_SIZE, IMG_SIZE, 3))
    
    # Wrapper
    encoded = layers.TimeDistributed(base_model)(inputs)
    
    # Bidirectional LSTM
    # Looks at the video Forward AND Backward
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=False))(encoded)
    
    # Classification Head
    x = layers.Dropout(0.5)(x) # Higher dropout to prevent overfitting
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid', dtype='float32')(x)
    
    return models.Model(inputs, outputs)

# WIPE MEMORY
tf.keras.backend.clear_session()

model = build_pro_model()

# Use a slightly higher LR initially for the LSTM, then lower it if stuck
# 0.00005 is a safe "middle ground" for fine-tuning
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.00005),
              loss='binary_crossentropy',
              metrics=['accuracy'])

I0000 00:00:1769926240.574075     235 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1769926240.579571     235 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [ ]:
# 7. TRAIN WITH CALLBACKS
# EarlyStopping stops training if it stops improving
# ReduceLROnPlateau lowers learning rate if accuracy gets stuck
callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(monitor='loss', factor=0.5, patience=2, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor='loss', patience=5, restore_best_weights=True)
]

In [16]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 10, 224, 224,   │             0 │
│                                 │ 3)                     │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 10, 2048)       │    23,564,800 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 256)            │     2,229,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25,827,073 (98.52 MB)

 Trainable params: 18,351,361 (70.00 MB)

 Non-trainable params: 7,475,712 (28.52 MB)

In [ ]:
tf.keras.backend.clear_session() 
print(" Memory wiped. Starting fresh...")

 Memory wiped. Starting fresh...


In [ ]:
print("STARTING PRO TRAINING...")
model.fit(
    train_gen,
    epochs=15, # Increased epochs (early stopping will handle it)
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=True
)

🚀 STARTING PRO TRAINING...


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/15


I0000 00:00:1769926354.289668     284 cuda_dnn.cc:529] Loaded cuDNN version 91002


778/778 ━━━━━━━━━━━━━━━━━━━━ 563s 592ms/step - accuracy: 0.7364 - loss: 0.5879 - learning_rate: 5.0000e-05
Epoch 2/15
778/778 ━━━━━━━━━━━━━━━━━━━━ 457s 587ms/step - accuracy: 0.9223 - loss: 0.1945 - learning_rate: 5.0000e-05
Epoch 3/15
778/778 ━━━━━━━━━━━━━━━━━━━━ 458s 588ms/step - accuracy: 0.9549 - loss: 0.1071 - learning_rate: 5.0000e-05
Epoch 4/15
778/778 ━━━━━━━━━━━━━━━━━━━━ 457s 587ms/step - accuracy: 0.9720 - loss: 0.0716 - learning_rate: 5.0000e-05
Epoch 5/15
778/778 ━━━━━━━━━━━━━━━━━━━━ 458s 588ms/step - accuracy: 0.9805 - loss: 0.0479 - learning_rate: 5.0000e-05
Epoch 6/15
778/778 ━━━━━━━━━━━━━━━━━━━━ 458s 588ms/step - accuracy: 0.9790 - loss: 0.0446 - learning_rate: 5.0000e-05
Epoch 7/15
778/778 ━━━━━━━━━━━━━━━━━━━━ 458s 589ms/step - accuracy: 0.9896 - loss: 0.0355 - learning_rate: 5.0000e-05
Epoch 8/15
778/778 ━━━━━━━━━━━━━━━━━━━━ 458s 588ms/step - accuracy: 0.9948 - loss: 0.0145 - learning_rate: 5.0000e-05
Epoch 9/15
778/778 ━━━━━━━━━━━━━━━━━━━━ 453s 581ms/step - accuracy:

In [ ]:
print("STARTING TRAINING...")
model.fit(train_gen, epochs=10, class_weight=class_weights,verbose=True)

In [ ]:
save_path = "/kaggle/working/deepfake_model.h5"
model.save(save_path)